<div align="center">

# Hafta 1: Python ve kardiyoloji veri temelleri

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/lab.html) · [PDF ders notu](Hafta01_Ders_Notlari.pdf) · [Colab'da aç (yedek yol)](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/NB01_python_ve_ekg_temelleri.ipynb)

</div>

**Bu Jupyter defteri nasıl kullanılır?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Bölüm 0 ortamı hazırlar ve ilk önce çalıştırılmalıdır. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar, bunların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. Deftere ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır.

**Üç yol.** Etiketsiz hücrelerin tümü **temel yolu** oluşturur ve hiç kod yazmamış olanlar dâhil herkes için yeterlidir. **İleri yol** etiketli hücreler, programlama bilenler için isteğe bağlı genişletmelerdir. Üçüncü yolda kod yazmak yerine bir yapay zekâ (YZ) asistanına prompt (istem) yazılır: **Prompt yolunda** her alıştırmanın altında hazır promptu içeren katlanmış bir *Prompt yolu* hücresi bulunur ve her bölüm bir *Prompt görevi* ile biter. Promptu asistana kopyalayın, alıştırma hücresinin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın; sonucun doğru olup olmadığını denetim satırı gösterir. Haftanın [prompt kütüphanesi](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md) bütün promptları bir araya getirir; [prompt yolu rehberi](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/PROMPT_PATH.tr.md) ise bu defterle çalışan asistanları anlatmaktadır.

**Nerede çalışır?** Defter, kendi bilgisayarınızdaki Anaconda içinde Jupyter için hazırlanmıştır (ders klasöründeki SETUP.tr.md dosyasına bakın). Anaconda kurulamıyorsa aynı defter, en üstteki bağlantıyla Google Colab'da açılabilir. Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda ve kurumunuzun kurallarıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan bir kez indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
COURSE_FILES = [
    "vclinic/__init__.py", "vclinic/ecg.py", "vclinic/cohort.py",
    "vclinic/measure.py", "vclinic/physionet.py", "vclinic/datasets.py",
    "vclinic/teach.py",
    "data/uci-heart-disease/processed.cleveland.data",
    "data/physionet/mitdb/100.hea", "data/physionet/mitdb/100.dat",
    "data/physionet/mitdb/100.atr",
    "data/physionet/ptbdb/patient001/s0010_re.hea",
    "data/physionet/ptbdb/patient001/s0010_re.dat",
    "data/physionet/ptbdb/patient001/s0010_re.xyz",
    "data/examples/example_caliper_measurements.csv",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None:
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course"
    for rel in COURSE_FILES:
        target = ROOT / rel
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026

print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["wfdb", "neurokit2", "ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 1.1": (0.833, 0.001),
    "Alıştırma 1.2": (439.1, 0.2),
    "Alıştırma 1.3": (2, 0),
    "Alıştırma A": (56.6, 0.1),
    "Alıştırma B": (32.0, 0.1),
    "Alıştırma C": (47, 0),
    "Alıştırma D": (100, 0),
    "Alıştırma E": (0.104, 0.002),
    "Alıştırma F": (403.9, 0.3),
    "Alıştırma G": (216, 6),
    "Alıştırma H": (33, 0),
    "Alıştırma I": (39.05, 0.1),
    "Alıştırma J": (466.5, 0.2),
})
print("Beklenen cevaplar hazır.")

## Python adımı 1: Değerler, listeler, sözlükler, koşullar, döngüler ve fonksiyonlar

*Ders: [Python adımı 1: Kliniğin dili](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#python-step)*

Python bu eğitimin dilidir. Bir kod hücresi Python komutlarını tutar ve Shift+Enter bu komutları çalıştırır. `#` işaretinden sonra gelen metin bir yorumdur: Python onu atlar, okura ise kodu açıklar. Bu adımın her hücresi `# Python alıştırması: dili öğrenmek için küçük bir örnek` satırıyla başlar. Örnekler, haftanın vakasındaki sayıları kullanmaktadır: Kalp hızı dakikada 85 atım (bpm) ve QT aralığı 391 ms olan 67 yaşında bir kadın hasta.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
heart_rate = 85          # dakikadaki atım (bpm)
qt_ms = 391              # milisaniye cinsinden QT aralığı
rr_s = 60 / heart_rate   # saniye cinsinden RR aralığı
print(rr_s)
print(round(rr_s, 3))

**Python notu.** `heart_rate = 85` bir *değişken* oluşturur: Bir değer tutan addır. `=` işareti atama yapar, eşitliği sınamaz. `/` bölme işlemidir ve `60 / heart_rate` RR aralığını saniye cinsinden verir. `print()` bir değeri gösterir, `round(değer, 3)` ise onu üç ondalığa yuvarlar. Python kayan noktalı sayılarla hesap yaptığı için `0.7058823529411765` yuvarlanana kadar tam hâliyle yazılır. Python'da ondalık ayırıcı her zaman noktadır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
patient_id = "VC-0317"
print(f"Hasta {patient_id}: {heart_rate} bpm, RR {rr_s:.3f} s")

**Python notu.** Tırnak içindeki metin bir *dizgedir* (string). `f` ile başlayan dizge bir *f-dizgesidir*: Süslü parantezlerin içindeki değerler metne yerleştirilir ve `{rr_s:.3f}` sayıyı üç ondalıkla yazar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
troponin = [3.6, 4.1, 3.9]          # seri ölçümler (ng/L)
print(troponin[0], troponin[-1], len(troponin))
troponin.append(4.0)                # dördüncü ölçüm geliyor
print(troponin, "max:", max(troponin))

**Python notu.** Köşeli parantezler bir *liste* oluşturur: Sıralı bir değer dizisi. Python konumları 0'dan saymaya başlar; bu yüzden `troponin[0]` ilk değer, `troponin[-1]` ise son değerdir. `len()` eleman sayısını verir, `.append()` sona bir eleman ekler, `max()` en büyük değeri döndürür.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
patient = {"id": "VC-0317", "age": 67, "sex": "F", "hr_bpm": 85, "qt_ms": 391}
print(patient["age"])
patient["lvef"] = 63                # yeni alan: sol ventrikül ejeksiyon fraksiyonu (%)
print(patient)

**Python notu.** `anahtar: değer` çiftleri içeren süslü parantezler bir *sözlük* (dictionary) oluşturur. Sözlük bir hasta dosyası gibi çalışır: Her alanın bir etiketi vardır. `patient["age"]` bir alanı okur, `patient["lvef"] = 63` ise yeni bir alan ekler.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
qtc_ms = 465
limit = 460 if patient["sex"] == "F" else 450
if qtc_ms >= limit:
    print(f"QTc {qtc_ms} ms ölçümü şu sınırda ya da üstünde: {limit} ms")
else:
    print(f"QTc {qtc_ms} ms ölçümü şu sınırın altında: {limit} ms")

**Python notu.** `if`, altındaki girintili satırları yalnızca koşul doğru olduğunda çalıştırır; `else` öteki durumu karşılar. Girinti (dört boşluk) hangi satırın hangi dala ait olduğunu gösterir. `==` eşitliği sınar, `>=` ise "büyük ya da eşit" anlamına gelir. `limit = 460 if ... else 450` satırı iki değerden birini tek satırda seçer. Bu iki sınır, Amerikan Kalp Derneği (American Heart Association, AHA) ve ortaklarının 2009 önerilerinde kadınlar ve erkekler için uzamış QTc eşikleridir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
clinic = [
    {"id": "VC-0317", "hr": 85, "qt": 391},
    {"id": "VC-0079", "hr": 58, "qt": 404},
    {"id": "VC-0254", "hr": 98, "qt": 343},
]
for p in clinic:
    rr = 60 / p["hr"]
    print(p["id"], round(rr, 3))

**Python notu.** `clinic`, her hasta için bir sözlük içeren bir listedir. `for p in clinic:` hastaları tek tek alır ve girintili satırlar her hasta için bir kez çalışır; tıpkı bir servis vizitesi gibi.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
def qtc_bazett(qt_ms, rr_s):
    """Bazett formülüyle kalp hızına göre düzeltilmiş QT aralığı (ms)."""
    return qt_ms / rr_s ** 0.5


print(round(qtc_bazett(391, 60 / 85), 1))

**Python notu.** `def` bir *fonksiyon* tanımlar: Tekrar kullanılabilen, adı olan bir işlem; bir klinik protokole benzer. `qt_ms` ve `rr_s` fonksiyonun girdileridir, `return` ise sonucu geri verir. Üç tırnak içindeki metin, fonksiyonun kısa açıklamasıdır (docstring). `**` üs alır; bu yüzden `rr_s ** 0.5` karekök demektir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
rr = np.array([0.81, 0.85, 0.83])   # üç RR aralığı (s)
print(60 / rr)                      # her atımın kalp hızı (bpm)
print(rr.mean(), rr.std(ddof=1))

**Python notu.** Bölüm 0'da `np` adıyla yüklenen NumPy, bütün bir sayı dizisi üzerinde tek seferde çalışır: `60 / rr`, 60'ı her RR aralığına böler. `.mean()` ortalamayı, `.std(ddof=1)` ise $n-1$'e bölen örneklem standart sapmasını verir.

**Hızlı kontrol.** `troponin.append(4.0)` satırından sonra `troponin[-1]` neyi döndürür?

<details><summary>Cevap</summary><code>4.0</code>: <code>-1</code> indeksi her zaman son elemanı verir ve <code>append()</code>, <code>4.0</code> değerini sona eklemiştir.</details>

**Alıştırma 1.1.** Dakikada 72 atımlık bir kalp hızı için RR aralığını saniye cinsinden hesaplayın ve üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma 1.1: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: Dakikada 72 atımlık (bpm) bir kalp hızı.
Görev: RR aralığını saniye cinsinden RR = 60 / HR formülüyle hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: RR aralığını üç ondalığa yuvarla. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 1.1", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 60 bpm için RR aralığını da yazdır; tam olarak 1,0 s olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 1.1", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(60 / 72, 3)
teach.check("Alıştırma 1.1", answer)

**Alıştırma 1.2.** QT aralığını RR aralığının küp köküne bölen bir `qtc_fridericia(qt_ms, rr_s)` fonksiyonu yazın: $QT_{cF} = QT / RR^{1/3}$. Fonksiyonu haftanın vakası için kullanın (QT 391 ms, kalp hızı 85 bpm) ve sonucu bir ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma 1.2: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: Milisaniye cinsinden QT aralığı ve saniye cinsinden RR aralığı. Haftanın vakasında kalp hızı 85 bpm iken QT 391 ms'dir; dolayısıyla RR = 60 / 85 s.
Görev: Fridericia formülüyle (QTc = QT / RR^(1/3)) düzeltilmiş QT aralığını döndüren bir `qtc_fridericia(qt_ms, rr_s)` fonksiyonu yaz ve fonksiyonu haftanın vakasına uygula.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Vakanın QTc değerini bir ondalığa yuvarla. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 1.2", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: assert ifadeleriyle denetle: `qtc_fridericia(400, 1.0)` 400,0, `qtc_fridericia(400, 0.729)` ise yuvarlandıktan sonra 444,4 döndürmeli (0,729 = 0,9³).
</pre>

</details>

In [ ]:
def qtc_fridericia(qt_ms, rr_s):
    return None   # None yerine formülü yazın


answer = qtc_fridericia(391, 60 / 85)
answer = round(answer, 1) if answer is not None else None
teach.check("Alıştırma 1.2", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
def qtc_fridericia(qt_ms, rr_s):
    """Fridericia formülüyle düzeltilmiş QT aralığı (ms)."""
    return qt_ms / rr_s ** (1 / 3)


answer = round(qtc_fridericia(391, 60 / 85), 1)
teach.check("Alıştırma 1.2", answer)

**Alıştırma 1.3.** `clinic` listesinde kalp hızı 80 bpm'nin üzerinde olan hastaları sayın. Bir `for` döngüsü ve bir `if` kullanın.

<details><summary><b>Prompt yolu</b>, Alıştırma 1.3: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `clinic` adlı liste her hastayı bir sözlük olarak tutar. Sözlüklerde "id", "hr" (bpm cinsinden kalp hızı) ve "qt" (ms cinsinden QT aralığı) anahtarları bulunur; örneğin {"id": "VC-0317", "hr": 85, "qt": 391}.
Görev: Kalp hızı 80 bpm'nin üstünde olan hastaları say.
Kısıtlar: pandas kullanmadan bir for döngüsü ve bir if ifadesi kullan. Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Hasta sayısı. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 1.3", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Kalp hızları 72, 85 ve 81 olan bir listede aynı döngü 2 hasta saymalıdır.
</pre>

</details>

In [ ]:
count = 0
# döngünüz buraya
answer = None   # cevabınız
teach.check("Alıştırma 1.3", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
count = 0
for p in clinic:
    if p["hr"] > 80:
        count = count + 1
answer = count
teach.check("Alıştırma 1.3", answer)

## 1. Hasta kaydı: pandas ile UCI Heart Disease verisi

*Ders: [pandas ile hasta tabloları](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#patient-tables-with-pandas)*

UCI Heart Disease koleksiyonunun Cleveland verisi, koroner anjiyografi yapılmış 303 hastanın 14 değişkenini içermektedir. `num` sütunu hastalığı 0'dan (yüzde 50'den fazla daralma yok) 4'e kadar derecelendirir. Bu bölüm haftanın ders içi mini projesidir: Veriyi yükleyin, özetleyin ve ilk şekilleri çizin.

**Formüllerden koda.** $\bar{x}$ ortalaması `df["chol"].mean()` ile hesaplanır. Paydasında $n-1$ bulunan $s$ örneklem standart sapması `df["chol"].std()` ile bulunur. Ortanca `.median()`, çeyrekler arası açıklık ise `.quantile(0.75) - .quantile(0.25)` ile hesaplanır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
from vclinic.datasets import load_cleveland

heart = load_cleveland()
print(heart.shape)          # (satır, sütun)
heart.head()

**Python notu.** `pd` adıyla yüklenen pandas, bir tabloyu *DataFrame* içinde tutar: Satırlar hastalar, sütunlar değişkenlerdir. `load_cleveland()`, paketle gelen dosyayı `pd.read_csv` ile okur ve sütunlara ad verir. `.shape` satır ve sütun sayısını, `.head()` ise ilk beş satırı gösterir.

| Sütun | Anlamı |
|---|---|
| `age`, `sex` | Yıl olarak yaş; cinsiyet (1 = erkek, 0 = kadın) |
| `cp` | Göğüs ağrısı tipi (1 tipik anjina, 2 atipik anjina, 3 anjina dışı ağrı, 4 asemptomatik) |
| `trestbps` | İstirahat sistolik kan basıncı (mmHg) |
| `chol` | Serum kolesterolü (mg/dL) |
| `fbs` | Açlık kan şekeri 120 mg/dL üzerinde (1 = evet) |
| `restecg` | İstirahat EKG'si (0 normal, 1 ST-T anormalliği, 2 sol ventrikül hipertrofisi) |
| `thalach` | Efor testinde ulaşılan en yüksek kalp hızı (bpm) |
| `exang` | Eforla ortaya çıkan anjina (1 = evet) |
| `oldpeak` | İstirahate göre eforla oluşan ST çökmesi (mm) |
| `slope` | Doruk eforda ST segmentinin eğimi (1 yukarı, 2 düz, 3 aşağı) |
| `ca` | Floroskopiyle boyanan ana damar sayısı (0 ile 3 arası) |
| `thal` | Talyum sintigrafisi (3 normal, 6 sabit defekt, 7 geri dönüşlü defekt) |
| `num`, `disease` | Anjiyografik hastalık derecesi 0 ile 4 arası; `num` > 0 ise `disease` = 1 |

In [ ]:
# Dersteki çözümlü örnek: Beş kolesterol değeri
chol5 = heart["chol"].head(5)
print(list(chol5))
mean = chol5.sum() / len(chol5)
sd = (((chol5 - mean) ** 2).sum() / (len(chol5) - 1)) ** 0.5
print(f"elle ortalama: {mean:.2f}  pandas: {chol5.mean():.2f}")
print(f"elle standart sapma: {sd:.2f}  pandas: {chol5.std():.2f}")
print(f"ortanca: {chol5.median():.1f}")

In [ ]:
# Hastalık durumuna göre temel değişkenlerin özeti
cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
print(heart["disease"].value_counts())
heart.groupby("disease")[cols].agg(["mean", "std"]).round(1)

**Python notu.** `value_counts()` bir sütundaki her değeri sayar. `groupby("disease")` tabloyu hastalığı olan ve olmayan hastalar olarak ikiye ayırır, `.agg(["mean", "std"])` ise her grup ve her sütun için bu iki istatistiği hesaplar.

In [ ]:
# Kaydın ilk iki şekli
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for value, color, label in [(0, "#1F5F8B", "hastalık yok"),
                            (1, "#9B1B1B", "hastalık var")]:
    sub = heart[heart["disease"] == value]
    axes[0].hist(sub["age"], bins=15, alpha=0.6, color=color, label=label)
axes[0].set_xlabel("Yaş (yıl)")
axes[0].set_ylabel("Hasta sayısı")
axes[0].legend()
heart.boxplot(column="thalach", by="disease", ax=axes[1])
axes[1].set_title("")
axes[1].set_xlabel("Hastalık (0 yok, 1 var)")
axes[1].set_ylabel("En yüksek kalp hızı (bpm)")
fig.suptitle("")
plt.tight_layout()
plt.show()

**Alıştırma A.** Hastalığı olan hastaların (`disease == 1`) ortalama yaşı nedir? Bir ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma A: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `heart` adlı pandas DataFrame'i, UCI Heart Disease veri setinin Cleveland verilerini her satırda bir hasta olacak biçimde tutar. age sütunu yaşı yıl olarak verir; disease sütunu koroner arter hastalığı varsa 1, yoksa 0'dır.
Görev: disease değeri 1 olan hastaların ortalama yaşını hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Ortalama yaşı bir ondalığa yuvarla. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma A", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Her hastalık grubundaki hasta sayısını yazdır; iki sayının toplamı 303 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(heart.loc[heart["disease"] == 1, "age"].mean(), 1)
teach.check("Alıştırma A", answer)

**Alıştırma B.** 303 hastanın yüzde kaçı kadındır (`sex == 0`)? Bir ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma B: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: sex sütunu erkekler için 1, kadınlar için 0 olarak kodlanmış `heart` DataFrame'i.
Görev: Kadınların bütün hastalar içindeki yüzdesini hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Yüzdeyi bir ondalığa yuvarla. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma B", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Erkeklerin yüzdesini de aynı biçimde hesapla ve iki yüzdenin toplamını yazdır; toplam 100 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma B", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(100 * (heart["sex"] == 0).mean(), 1)
teach.check("Alıştırma B", answer)

**Gözlemle ve düşün.** Özet tablosunda iki grup arasında en çok hangi değişken farklılaşıyor? Farkın yönü klinik açıdan akla yatkın mı? Kadın hasta sayısının azlığı, verinin ne kadar genellenebileceği hakkında ne söylüyor?

**İleri yol.** İki sütunda eksik değer vardır. Bunları `heart.isna().sum()` ile sayın, ardından göğüs ağrısı tipine göre hastalık oranını `pd.crosstab(heart["cp"], heart["disease"], normalize="index")` ile karşılaştırın.

In [ ]:
# İleri yol
print(heart.isna().sum()[heart.isna().sum() > 0])
pd.crosstab(heart["cp"], heart["disease"], normalize="index").round(2)

**Prompt görevi, 1. bölüm: Hastalık durumuna göre kolesterol.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: chol (mg/dL cinsinden serum kolesterolü) ve disease (koroner arter hastalığı için 1, yoksa 0) sütunlarını içeren `heart` DataFrame'i.
Görev: İki hastalık grubunun chol değerleri için yan yana iki kutu grafiği çiz; eksen adlarını birimleriyle yaz. Her grubun ortanca kolesterol değerini yazdır.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve iki ortanca değer. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Her gruptaki hasta sayısını yazdır; iki sayının toplamı 303 olmalı.
</pre>

</details>

**Beklenen sonuç.** Ortanca değer hastalığı olmayanlarda 234,5 mg/dL, hastalığı olanlarda 249,0 mg/dL'dir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 2. Sanal klinik açılıyor

*Ders: [Sanal kardiyoloji kliniği, eğitimin projesi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-virtual-cardiology-clinic-the-project-of-the-course)*

`vclinic.make_cohort()` sanal kardiyoloji kliniğinin hasta kaydını oluşturur. Her hastanın demografik verileri, risk faktörleri, laboratuvar değerleri, sol ventrikül ejeksiyon fraksiyonu (LVEF), ana tanısı ve istirahat 12 derivasyonlu elektrokardiyogramının (EKG) parametreleri vardır. Bir hasta eğitimin her haftasında aynı değerleri taşır. Veriler sentetiktir: Genel klinik örüntüleri izlerler ama gerçek bir kişiyi tanımlamazlar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
clinic = vclinic.make_cohort(n=500, seed=2026)
print(clinic.shape)
clinic.head()

**Python notu.** `vclinic.make_cohort(n=500, seed=2026)`, ders paketinin bir fonksiyonunu iki *anahtar sözcüklü argümanla* çağırır. Tohum değeri (seed) rastgele sayıları sabitler; böylece her çalıştırma aynı 500 hastayı verir.

In [ ]:
# Kliniğin ana tanıları ve ritmi
print(clinic["main_diagnosis"].value_counts())
print(clinic["ecg_rhythm"].value_counts())
clinic[["age", "bmi", "sbp", "ldl", "hdl", "egfr", "lvef", "ntprobnp",
        "ecg_hr", "ecg_pr", "ecg_qrs", "ecg_qt"]].describe().round(1)

Tanı kodları uluslararası kısaltmalardır: `CCS` kronik koroner sendrom (KKS), `ACS` akut koroner sendrom (AKS), `HFrEF` ejeksiyon fraksiyonu düşük kalp yetersizliği, `HFpEF` ejeksiyon fraksiyonu korunmuş kalp yetersizliği ve `none` yapısal kalp hastalığı yok demektir. Ritim sütunundaki `AF` atriyal fibrilasyondur. Veri sözlüğünün tamamı DATASETS.tr.md dosyasındadır.

In [ ]:
# Sanal klinik ve Cleveland verisi yan yana
compare = pd.DataFrame({
    "Cleveland (gerçek)": [
        heart["age"].mean(), 100 * heart["sex"].mean(),
        heart["trestbps"].mean(), heart["chol"].mean(),
        100 * heart["disease"].mean()],
    "Sanal klinik (sentetik)": [
        clinic["age"].mean(), 100 * (clinic["sex"] == "M").mean(),
        clinic["sbp"].mean(), clinic["total_chol"].mean(),
        100 * clinic["cad"].mean()],
}, index=["Ortalama yaş (yıl)", "Erkek (%)",
          "Sistolik basınç (mmHg)",
          "Total kolesterol (mg/dL)",
          "Koroner hastalık (%)"])
compare.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(heart["age"], bins=20, density=True, alpha=0.55, color="#1F5F8B",
        label="Cleveland (gerçek)")
ax.hist(clinic["age"], bins=20, density=True, alpha=0.55, color="#9B1B1B",
        label="Sanal klinik (sentetik)")
ax.set_xlabel("Yaş (yıl)")
ax.set_ylabel("Yoğunluk")
ax.legend()
plt.tight_layout()
plt.show()

**Alıştırma C.** Sanal klinikte ana tanısı `HFrEF` olan kaç hasta vardır?

<details><summary><b>Prompt yolu</b>, Alıştırma C: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `clinic` adlı bir pandas DataFrame'i sanal kliniğin 500 sentetik hastasını tutar. main_diagnosis sütunu "none", "CCS", "HFpEF", "HFrEF" ve "ACS" etiketlerinden birini içerir.
Görev: Ana tanısı tam olarak "HFrEF" olan hastaları say.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Hasta sayısı, tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma C", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Bütün tanıların sayılarını value_counts() ile yazdır; toplamları 500 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma C", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = int((clinic["main_diagnosis"] == "HFrEF").sum())
teach.check("Alıştırma C", answer)

**Gözlemle ve düşün.** İki kohort tabloda birbirine benziyor. Yine de sentetik bir kayıt, üzerinde eğitilecek bir model açısından önemli olabilecek hangi yönlerden gerçeğinden ayrılabilir? Nadir durumları, değişkenler arasındaki ilişkileri ve ölçüm gürültüsünü düşünün.

**Prompt görevi, 2. bölüm: Cinsiyete göre tanılar.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: Sanal kliniğin `clinic` DataFrame'i; main_diagnosis sütununu ve "F" ya da "M" olarak kodlanmış sex sütununu içerir.
Görev: pd.crosstab ile her ana tanı için cinsiyete göre hasta sayılarını gösteren, satır ve sütun toplamları olan bir tablo oluştur.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Tablo. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Genel toplam 500 olmalı.
</pre>

</details>

**Beklenen sonuç.** Klinikte 221 kadın ve 279 erkek vardır. HFrEF tanılı 47 hastanın 16'sı kadın, 31'i erkektir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 3. Veri olarak EKG: Örnekler, örnekleme frekansı ve birimler

*Ders: [Veri olarak EKG](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-ecg-as-data)*

**Formüllerden koda.** $f_s$ örnekleme frekansı `case.fs` içindedir. $n$ numaralı örneğin zamanı, yani $t_n = n / f_s$, `case.time` içinde tutulur. Bir aralıktaki örnek sayısı $N = \Delta t \cdot f_s$ ise `round(dt * case.fs)` ile hesaplanır. Milivolt cinsinden gerilimler `case.leads` tablosunun sütunlarıdır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
case = vclinic.patient_ecg(clinic, "VC-0317")   # haftanın vakası
print(case)
print("derivasyon başına örnek sayısı:", len(case.leads))
print("zaman adımı (s):", 1 / case.fs)
case.leads.head()

In [ ]:
# EKG kâğıdında D2 derivasyonu: 10 saniye ve tek atım
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5))
teach.plot_strip(axes[0], case.time, case.lead("II"), label="II")
one = (case.time >= 1.0) & (case.time <= 2.0)
teach.plot_strip(axes[1], case.time[one], case.lead("II")[one], label="II")
axes[1].plot(case.time[one][::10], case.lead("II")[one][::10], "o",
             color="#9B1B1B", ms=3, label="her 10. örnek")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

**Python notu.** `case.time >= 1.0` her zaman değerini `1.0` ile karşılaştırır ve `True` ile `False` değerlerinden oluşan bir dizi, yani bir *maske* döndürür. `&` ile birleştirilen iki maske 1 ile 2 saniye arasındaki örnekleri tutar ve `x[mask]` bunları seçer. `[::10]` her onuncu elemanı alır.

In [ ]:
# Üç örnekleme frekansında bir RR aralığındaki örnek sayısı
rr = 60 / 85
for fs in [500, 360, 1000]:
    print(f"fs = {fs} Hz: RR = {rr:.3f} s şu kadar örnek içerir: {round(rr * fs)} örnek")

**Keşfet.** Panel, vakanın bir kalp atımını, bir analog-dijital dönüştürücünün (ADC) yapacağı gibi daha düşük bir frekansta yeniden örnekler ve daha az bitle saklar. Referans, aynı atımın 2000 Hz'lik hâlidir. Kaydırıcıları hareket ettirin ve R tepesinin yüksekliğinin ve zamanının nasıl değiştiğine bakın. Yaklaşık 100 Hz'in altında R tepesi kırpılır ve kayar. Bit sayısı azaldıkça çizgi basamaklara dönüşür.

In [ ]:
# Keşfet: örnekleme frekansı ve çözünürlük
fine = vclinic.simulate(case.params, duration_s=2.0, fs=2000, noise="none", seed=7)
t_ref, x_ref = fine.time, fine.lead("II")


def show_sampling(fs_hz=500, bits=12):
    step = int(round(2000 / fs_hz))
    t_s, x_s = t_ref[::step], x_ref[::step]
    q = 10.0 / 2 ** bits                       # 10 mV aralıkta adım büyüklüğü
    x_q = np.round(x_s / q) * q
    window = (t_ref > 0.25) & (t_ref < 0.75)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(t_ref[window], x_ref[window], color="#888888", lw=1,
            label="2000 Hz referans")
    w2 = (t_s > 0.25) & (t_s < 0.75)
    ax.step(t_s[w2], x_q[w2], where="mid", color="#9B1B1B", lw=1.2,
            label=f"{fs_hz} Hz, {bits} bit")
    ax.set_xlabel("Zaman (s)")
    ax.set_ylabel("mV")
    ax.legend(loc="upper right")
    plt.show()
    print(f"R tepesi: referans {x_ref.max():.3f} mV, "
          f"örneklenmiş {x_q.max():.3f} mV; "
          f"zaman kayması {abs(t_s[np.argmax(x_q)] - t_ref[np.argmax(x_ref)]) * 1000:.1f} ms; "
          f"çözünürlük {q * 1000:.2f} µV")


try:
    from ipywidgets import interact, IntSlider
    interact(show_sampling,
             fs_hz=IntSlider(value=500, min=50, max=1000, step=50, description="fs (Hz)"),
             bits=IntSlider(value=12, min=4, max=16, step=1, description="bits"))
except ImportError:
    show_sampling(500, 12)
    show_sampling(100, 6)

**Alıştırma D.** EKG kâğıdında bir büyük kare 0,2 s'ye karşılık gelir. 500 Hz'de bir büyük kare kaç örnek içerir?

<details><summary><b>Prompt yolu</b>, Alıştırma D: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: 500 Hz'de örneklenmiş bir EKG. EKG kâğıdında büyük bir kare 0,2 s'ye karşılık gelir.
Görev: Büyük bir karenin kaç örnek içerdiğini N = süre × örnekleme frekansı ile hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Örnek sayısı, tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma D", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Aynı örnekleme frekansında 0,04 s'lik küçük bir kare 20 örnek içermelidir.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma D", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(0.2 * 500)
teach.check("Alıştırma D", answer)

**Prompt görevi, 3. bölüm: D2'nin bir saniyesi, örnek örnek.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `case`, haftanın vakasının EKG'sidir: case.time saniye cinsinden zamanları, case.lead("II") D2 derivasyonunun milivolt cinsinden voltajlarını, case.fs ise Hz cinsinden örnekleme frekansını tutar.
Görev: D2'nin ilk saniyesini, her örneği küçük bir işaretle göstererek çiz; zaman ekseni milisaniye, voltaj ekseni milivolt cinsinden olsun. Çizilen örnek sayısını yazdır.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve bir sayı. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 500 Hz'de bir saniye 500 örnek içerir.
</pre>

</details>

**Beklenen sonuç.** Yazdırılan sayı 500'dür (iki uç da dâhil edildiyse 501) ve zaman ekseni 0'dan 1000 ms'ye uzanır.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 4. Tek kalp vektöründen on iki derivasyon

*Ders: [Bir vektör olarak kalp: On iki derivasyon, tek kaynak](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-heart-as-a-vector-twelve-leads-one-source)*

**Formüllerden koda.** $\mathbf{H} = (H_x, H_y, H_z)$ kalp vektörü üç sayılık bir NumPy dizisidir. On iki derivasyonun $\mathbf{L}$ derivasyon vektörleri `vclinic.LEAD_MATRIX` matrisinin satırlarıdır. Bütün derivasyonlar için $V_L = \mathbf{L} \cdot \mathbf{H}$ gerilimi tek seferde `vclinic.project(H)` ile hesaplanır; bu fonksiyon `LEAD_MATRIX @ H` işlemini yapar. Frontal eksen $\alpha = \operatorname{atan2}(H_y, H_x)$ ise `vclinic.frontal_axis(V_I, V_aVF)` ile bulunur.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
# Sanal kliniğin derivasyon vektörleri: x sola, y aşağı, z öne
lead_table = pd.DataFrame(vclinic.LEAD_MATRIX, index=vclinic.LEADS,
                          columns=["Lx", "Ly", "Lz"]).round(3)
lead_table

In [ ]:
# Dersteki çözümlü örnek
H = np.array([0.6, 0.8, -0.2])           # kalp vektörü (mV)
V = vclinic.project(H)
volts = pd.Series(V, index=vclinic.LEADS).round(3)
print(volts.to_string())
print("Einthoven: I + III =", round(volts["I"] + volts["III"], 3), "ve II =", volts["II"])
print("Goldberger: aVR + aVL + aVF =", round(volts["aVR"] + volts["aVL"] + volts["aVF"], 3))
print("Frontal eksen (derece):", round(vclinic.frontal_axis(V[0], V[5]), 1))

**Python notu.** `@` bir matrisi bir vektörle çarpar: 12 × 3 boyutlu derivasyon matrisinin her satırı `H` içindeki üç sayıyla eleman eleman çarpılır ve çarpımlar toplanır. `pd.Series`, burada derivasyon adlarıyla etiketlenmiş tek bir sütundur.

In [ ]:
# Vakanın 12 derivasyonlu EKG'si
fig = teach.plot_12lead(case.time, case.leads, title="VC-0317")
plt.show()
L = case.leads
print("Einthoven yasasından en büyük sapma (mV):",
      float(np.abs(L["II"] - (L["I"] + L["III"])).max()))

**Keşfet.** Kalp vektörünü frontal düzlemde ve göğsün önüne ya da arkasına doğru döndürün ve on iki izdüşümü izleyin. Vektör bir derivasyonun yönünü gösterdiğinde o derivasyon en büyük pozitif sapmasını, ters yönü gösterdiğinde negatif bir sapma çizer.

In [ ]:
# Keşfet: tek vektör, on iki izdüşüm
def show_projection(frontal=50, anterior=-15, size=1.3):
    h = size * vclinic.direction(frontal, anterior)
    v = vclinic.project(h)
    colors = ["#9B1B1B" if x >= 0 else "#1F5F8B" for x in v]
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.bar(vclinic.LEADS, v, color=colors)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylim(-2.5, 2.5)
    ax.set_ylabel("İzdüşüm (mV)")
    ax.set_title(f"H = ({h[0]:.2f}, {h[1]:.2f}, {h[2]:.2f}) mV, eksen "
                 f"{vclinic.frontal_axis(v[0], v[5]):.0f}°")
    plt.show()


try:
    from ipywidgets import interact, IntSlider, FloatSlider
    interact(show_projection,
             frontal=IntSlider(value=50, min=-180, max=180, step=5, description="frontal (°)"),
             anterior=IntSlider(value=-15, min=-90, max=90, step=5, description="ön/arka (°)"),
             size=FloatSlider(value=1.3, min=0.2, max=2.0, step=0.1, description="büyüklük (mV)"))
except ImportError:
    show_projection(50, -15, 1.3)

**Alıştırma E.** $\mathbf{H} = (0{,}6;\ 0{,}8;\ -0{,}2)$ mV için aVL derivasyonunun gerilimini, `lead_table` içindeki aVL derivasyon vektörüyle hesaplayın ve üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma E: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `vclinic.LEAD_VECTORS`, her derivasyon adını ("I", "II", "III", "aVR", "aVL", "aVF" ve "V1"-"V6") üç sayılık bir NumPy dizisine eşleyen bir sözlüktür. x ekseni sola, y ekseni ayaklara, z ekseni öne doğrudur. Bir derivasyonun milivolt cinsinden voltajı, derivasyon vektörünün kalp vektörü H ile nokta çarpımıdır.
Görev: H = (0,6; 0,8; -0,2) mV için aVL derivasyonunun voltajını hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Voltajı üç ondalığa yuvarla ve float türüne çevir. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma E", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: D1 derivasyonunun (sözlükte "I") vektörü (1; 0; 0) olduğundan aynı kod D1 için 0,6 mV vermelidir.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma E", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
l_avl = vclinic.LEAD_VECTORS["aVL"]
answer = round(float(l_avl @ np.array([0.6, 0.8, -0.2])), 3)
teach.check("Alıştırma E", answer)

**Gözlemle ve düşün.** Aynı vektör D2 derivasyonunda pozitif, V1 derivasyonunda ise negatif bir sapma veriyor. Farkı, vektör ile her derivasyon arasındaki açıyla açıklayın. Aynı büyüklükteki vektörler için göğüs derivasyonları neden ekstremite derivasyonlarından daha büyük sapmalar gösterir?

**Prompt görevi, 4. bölüm: Bütün kliniğin frontal ekseni.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: Derece cinsinden frontal QRS eksenini tutan ecg_axis sütununu içeren `clinic` DataFrame'i.
Görev: ecg_axis değerlerinin histogramını çiz ve -30° ile +90°'ye dikey birer çizgi koy. Sol eksen sapması (eksen -30°'nin altında) ve sağ eksen sapması (eksen +90°'nin üstünde) olan hastaların sayısını ve yüzdesini yazdır.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve dört sayı. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Sol, normal ve sağ olmak üzere üç grubun sayılarının toplamı 500 olmalı.
</pre>

</details>

**Beklenen sonuç.** Altı hastada (yüzde 1,2) sol eksen sapması, 25 hastada (yüzde 5,0) sağ eksen sapması vardır.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 5. PR, QRS ve QT ölçümü

*Ders: [Aralıklar: PR, QRS ve QT](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#intervals-pr-qrs-and-qt)*

**Formüllerden koda.** $HR = 60 / RR$ kalp hızı `60 / rr_s` ile hesaplanır. Bazett düzeltmesi $QT_{cB} = QT / \sqrt{RR}$ için `qt / np.sqrt(rr_s)`, Fridericia düzeltmesi $QT_{cF} = QT / \sqrt[3]{RR}$ için ise `qt / np.cbrt(rr_s)` kullanılır; ikisinde de RR aralığı saniye cinsinden olmalıdır. Teğet yöntemiyle bulunan T dalgası sonu, $t_{end} = t_s + (b - x(t_s)) / x'(t_s)$, `t_end_tangent` fonksiyonunda `t_s + (base - x[t_s]) / slope * fs` satırıyla hesaplanır.

Bu bölümde ölçüm beş adımda kurulur ve adımlar sonunda tek bir fonksiyonda, `measure_ecg` içinde birleştirilir. Sonraki haftalarda kullanılmak üzere aynı adımlar `vclinic.measure` modülünde de yer almaktadır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
# Adım 1. R tepeleri: bant geçiren süzgeç, türev, kare alma ve 150 ms'lik hareketli pencere
from scipy.signal import butter, filtfilt, find_peaks


def bandpass(x, fs, low, high):
    b, a = butter(2, [low / (fs / 2), high / (fs / 2)], btype="band")
    return filtfilt(b, a, x)


def lowpass(x, fs, cutoff=20.0):
    b, a = butter(2, cutoff / (fs / 2), btype="low")
    return filtfilt(b, a, x)


def detect_r_peaks(x, fs):
    y = bandpass(x, fs, 5, 15)                 # QRS frekanslarını tutar
    d = np.gradient(y) * fs                    # eğim (mV/s)
    z = d ** 2                                 # yalnızca büyük eğimler öne çıkar
    w = int(round(0.150 * fs))                 # 150 ms'lik pencere
    m = np.convolve(z, np.ones(w) / w, mode="same")
    peaks, _ = find_peaks(m, height=0.3 * np.percentile(m, 99),
                          distance=int(0.25 * fs))
    yy = bandpass(x, fs, 1, 40)
    half = int(0.075 * fs)
    r = []
    for p in peaks:                            # 75 ms içindeki en büyük sapma
        lo, hi = max(0, p - half), min(len(x), p + half + 1)
        r.append(lo + int(np.argmax(np.abs(yy[lo:hi]))))
    return np.unique(r)


r_idx = detect_r_peaks(case.lead("II"), case.fs)
print("Bulunan R tepeleri:", len(r_idx))
print("zamanlar (s):", np.round(r_idx / case.fs, 3))
print("gerçek R zamanları (s):", np.round(case.truth["all_beats_s"], 3))

In [ ]:
# Adım 2. 12 derivasyonun eğiminden QRS başlangıcı ve sonu
def global_slope(leads, fs):
    arr = np.column_stack([bandpass(leads[:, j], fs, 1, 40)
                           for j in range(leads.shape[1])])
    g = np.abs(np.gradient(arr, axis=0)).sum(axis=1) * fs
    w = max(1, int(round(0.010 * fs)))
    return np.convolve(g, np.ones(w) / w, mode="same")


def qrs_bounds(leads, fs, r_idx, frac=0.08):
    g = global_slope(leads, fs)
    onsets, offsets = [], []
    for r in r_idx:
        lo, hi = max(0, r - int(0.12 * fs)), min(len(g) - 1, r + int(0.14 * fs))
        level = frac * g[lo:hi + 1].max()      # en büyük eğimin yüzde 8'i
        on = r
        while on > lo and g[on] > level:
            on -= 1
        off = r
        while off < hi and g[off] > level:
            off += 1
        onsets.append(on)
        offsets.append(off)
    return np.array(onsets), np.array(offsets)


on_idx, off_idx = qrs_bounds(case.leads.to_numpy(), case.fs, r_idx)
qrs_ms = (off_idx - on_idx) / case.fs * 1000
print("QRS süreleri (ms):", np.round(qrs_ms, 0))

**Python notu.** `while`, koşulu doğru kaldıkça girintili satırlarını tekrarlar. Burada döngü, R tepesinden başlayıp her adımda bir örnek geriye giderek 12 derivasyonun eğimi eşik düzeyinin altına düşene kadar ilerler. `np.column_stack` dizileri bir matrisin sütunları olarak yan yana koyar.

In [ ]:
# Adım 3. Teğet yöntemiyle T dalgasının sonu (D2, 20 Hz alçak geçiren süzgeçten geçmiş)
def t_end_tangent(x, fs, qrs_off, next_r, base):
    lo = qrs_off + int(0.08 * fs)
    hi = qrs_off + int(0.7 * (next_r - qrs_off))
    seg = x[lo:hi] - base
    k = int(np.argmax(np.abs(seg)))
    peak = lo + k                              # T tepesi
    sign = 1.0 if seg[k] >= 0 else -1.0
    d = np.gradient(x) * fs
    stop = min(len(x) - 1, peak + int(0.20 * fs))
    t_s = peak + int(np.argmin(sign * d[peak:stop]))   # en dik nokta
    slope = d[t_s]
    if slope == 0:
        return np.nan
    return t_s + (base - x[t_s]) / slope * fs  # teğetin taban çizgisini kestiği yer


x20 = lowpass(case.lead("II"), case.fs, 20)
fs = case.fs
i = 2                                          # üçüncü atım
base = np.median(x20[on_idx[i] - int(0.03 * fs):on_idx[i]])
t_end = t_end_tangent(x20, fs, off_idx[i], r_idx[i + 1], base)
print(f"QT atımı: {(t_end - on_idx[i]) / fs * 1000:.1f} ms, gerçek: {case.truth['qt_ms']:.0f} ms")

In [ ]:
# Üçüncü atımda teğet çizimi
d20 = np.gradient(x20) * fs
seg = x20[off_idx[i] + int(0.08 * fs):off_idx[i] + int(0.7 * (r_idx[i + 1] - off_idx[i]))]
peak = off_idx[i] + int(0.08 * fs) + int(np.argmax(np.abs(seg - base)))
t_s = peak + int(np.argmin(d20[peak:peak + int(0.2 * fs)]))
tt = np.linspace(t_s / fs - 0.05, t_end / fs + 0.02, 50)
fig, ax = plt.subplots(figsize=(9, 4))
w = slice(on_idx[i] - int(0.1 * fs), int(t_end) + int(0.12 * fs))
teach.plot_strip(ax, case.time[w], x20[w], label="II", v_range=(-0.5, 1.5))
ax.plot(tt, x20[t_s] + d20[t_s] * (tt - t_s / fs), color="#9B1B1B", lw=1.5,
        label="en dik noktadaki teğet")
ax.axhline(base, color="#1F5F8B", lw=1, ls="--", label="taban çizgisi")
ax.axvline(on_idx[i] / fs, color="k", lw=1, ls=":")
ax.axvline(t_end / fs, color="#9B1B1B", lw=1, ls=":")
ax.legend(loc="upper right", fontsize=8)
plt.show()

In [ ]:
# Adım 4. P dalgasının başlangıcı
def p_onset(x, fs, qrs_on, earliest=0, frac=0.10):
    lo = max(0, int(earliest), qrs_on - int(0.30 * fs))
    hi = qrs_on - int(0.04 * fs)
    if hi - lo < int(0.08 * fs):               # bir P dalgası için çok kısa
        return np.nan
    seg = x[lo:hi]
    base = np.median(seg[:int(0.04 * fs)])     # P dalgasından önceki düz bölüm
    seg = seg - base
    k = int(np.argmax(seg))
    j = k
    while j > 0 and seg[j] > frac * seg[k]:    # P yüksekliğinin yüzde 10'una kadar geri
        j -= 1
    return lo + j


po = p_onset(x20, fs, on_idx[i], earliest=t_end_tangent(
    x20, fs, off_idx[i - 1], r_idx[i], base) + 0.02 * fs)
print(f"PR atımı: {(on_idx[i] - po) / fs * 1000:.0f} ms, gerçek: {case.truth['pr_ms']:.0f} ms")

In [ ]:
# Adım 5. Bütün adımlar tek fonksiyonda: EKG laboratuvarının ölçümü
def measure_ecg(leads, fs, lead="II", sinus=True):
    """12 derivasyonlu bir kaydın ortanca kalp hızı, PR, QRS, QT ve QTc değerleri."""
    leads = np.asarray(leads, dtype=float)
    x = leads[:, ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3",
                  "V4", "V5", "V6"].index(lead)]
    r = detect_r_peaks(leads[:, 1], fs)
    on, off = qrs_bounds(leads, fs, r)
    x20 = lowpass(x, fs, 20)
    pr, qrs, qt = [], [], []
    t_prev = None
    for k in range(len(r) - 1):
        base = np.median(x20[max(0, on[k] - int(0.03 * fs)):on[k]])
        te = t_end_tangent(x20, fs, off[k], r[k + 1], base)
        qrs.append((off[k] - on[k]) / fs * 1000)
        qt.append((te - on[k]) / fs * 1000)
        if sinus and t_prev is not None and np.isfinite(t_prev):
            pr.append((on[k] - p_onset(x20, fs, on[k], t_prev + 0.02 * fs)) / fs * 1000)
        t_prev = te
    rr_s = np.mean(np.diff(r)) / fs
    out = {"hr_bpm": 60 / rr_s, "rr_s": rr_s,
           "pr_ms": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
           "qrs_ms": np.median(qrs), "qt_ms": np.nanmedian(qt)}
    out["qtc_bazett_ms"] = out["qt_ms"] / np.sqrt(rr_s)
    out["qtc_fridericia_ms"] = out["qt_ms"] / np.cbrt(rr_s)
    return out


def compare(ecg):
    m = measure_ecg(ecg.leads.to_numpy(), ecg.fs, sinus=ecg.truth["rhythm"] == "sinus")
    keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
    return pd.DataFrame({"ölçülen": [m[k] for k in keys],
                         "gerçek": [ecg.truth[k] for k in keys]},
                        index=keys).round(1)


compare(case)

**Gözlemle ve düşün.** Haftanın vakasında Bazett formülü, kadınlar için 460 ms olan eşiğin üstünde bir QTc vermektedir. Fridericia formülü ise bu eşiğin belirgin biçimde altında bir değer verir. Dakikada 85 atımda hangi düzeltmeyi rapora yazardınız ve neden? Dersin QT düzeltmesi bölümü, 2009 bildirgesinin önerisini vermektedir.

In [ ]:
# Etkileşimli labın C bölümündeki beş hasta
LAB_PATIENTS = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0018"]
rows = []
for pid in LAB_PATIENTS:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs, sinus=e.truth["rhythm"] == "sinus")
    rows.append({"patient": pid, "conduction": e.truth["conduction"],
                 "HR": e.truth["hr_bpm"], "PR_true": e.truth["pr_ms"], "PR_alg": m["pr_ms"],
                 "QRS_true": e.truth["qrs_ms"], "QRS_alg": m["qrs_ms"],
                 "QT_true": e.truth["qt_ms"], "QT_alg": m["qt_ms"]})
lab_table = pd.DataFrame(rows).set_index("patient").round(1)
lab_table

Listedeki son hasta olan VC-0018'de sol dal bloğu (left bundle branch block, LBBB) vardır. QRS kompleksi geniştir ve D2'deki T dalgası küçük olup ana QRS yönüne zıttır. Bu yüzden D2'de teğet yöntemi bu hasta için güvenilir değildir; aşağıdaki ileri yol nedenini göstermektedir.

In [ ]:
# Klinikten 100 hasta üzerinde doğrulama (sinüs ritmi, normal iletim)
sample = clinic[(clinic["ecg_rhythm"] == "sinus")
                & (clinic["ecg_conduction"] == "normal")].head(100)
errors, qt_pairs = [], []
for pid in sample["patient_id"]:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs)
    errors.append({k: m[k] - e.truth[k] for k in ["pr_ms", "qrs_ms", "qt_ms"]})
    qt_pairs.append((m["qt_ms"], e.truth["qt_ms"]))
errors = pd.DataFrame(errors)
qt_pairs = np.array(qt_pairs)
summary = pd.DataFrame({"sapma (ms)": errors.mean(),
                        "SS (ms)": errors.std(),
                        "error| (ms)|en büyük |hata| (ms)": errors.abs().max()}).round(1)
summary

**Sizin kumpasınız, algoritma ve gerçek değer.** Etkileşimli labın *Düşün ve dışa aktar* sekmesi, C bölümündeki kumpas ölçümlerinizi `my_caliper_measurements.csv` dosyası olarak dışa aktarır. Dosyayı bu defterin klasörüne kaydedin ve sonraki hücreyi çalıştırın. Dosya indirilemiyorsa *Kumpas ölçümlerini kopyala* düğmesi aynı metni kopyalar: Metni sonraki hücrede `PASTED_CALIPERS` değişkeninin üçlü tırnakları arasına yapıştırın. Kendi ölçümleriniz yoksa hücre, benzetimle üretilmiş elle ölçümlerden oluşan bir örnek dosya kullanır.

In [ ]:
# QT aralığının Bland-Altman analizi
import io
import os
from vclinic.datasets import data_path

# kopyalanan kumpas ölçümlerini üçlü tırnakların arasına yapıştırın
PASTED_CALIPERS = """
"""

own = "my_caliper_measurements.csv"
if PASTED_CALIPERS.strip():
    source = "PASTED_CALIPERS"
    calipers = pd.read_csv(io.StringIO(PASTED_CALIPERS.strip()))
elif os.path.exists(own):
    source = own
    calipers = pd.read_csv(own)
else:
    source = data_path("examples", "example_caliper_measurements.csv")
    calipers = pd.read_csv(source)
print("Ölçümlerin kaynağı:", source)
qt_manual = calipers[calipers["measure"] == "QT"].set_index("patient_id")["value_ms"]
pts = [p for p in qt_manual.index if p != "VC-0018"]
truth_qt = [vclinic.patient_ecg(clinic, p).truth["qt_ms"] for p in pts]
alg_qt = [lab_table.loc[p, "QT_alg"] for p in pts]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
teach.plot_bland_altman(axes[0], qt_manual[pts].to_numpy(), truth_qt)
axes[0].set_title("Kumpas eksi gerçek")
teach.plot_bland_altman(axes[1], qt_pairs[:, 0], qt_pairs[:, 1])
axes[1].set_title("Algoritma eksi gerçek, 100 hasta")
plt.tight_layout()
plt.show()

**Python notu.** `elif`, *değilse eğer* anlamına gelir: Python koşulları yukarıdan aşağıya sınar ve yalnızca koşulu doğru olan ilk bloğu çalıştırır. `io.StringIO` bir metni dosya gibi davranır hâle getirir; böylece `pd.read_csv`, yapıştırılan metni diskteki bir dosya gibi okuyabilir.

**Alıştırma F.** VC-0254 numaralı hastanın kayıttaki QT değeri, dakikada 98 atımlık kalp hızında 343 ms'dir. Bu iki sayıdan Fridericia formülüyle QTc değerini hesaplayın ve bir ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma F: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: Kalp hızı 98 bpm iken 343 ms'lik bir QT aralığı.
Görev: QTc'yi Fridericia formülüyle (QTc = QT / RR^(1/3)) hesapla; saniye cinsinden RR aralığını RR = 60 / HR ile bul.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: QTc'yi bir ondalığa yuvarla. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma F", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 60 bpm'de RR aralığı 1 s olduğundan QTc, QT'ye eşit olmalıdır.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma F", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(343 / np.cbrt(60 / 98), 1)
teach.check("Alıştırma F", answer)

**Alıştırma G.** VC-0188 numaralı hastanın PR aralığını `measure_ecg` ile ölçün ve en yakın milisaniyeye yuvarlayın. Sonuç birinci derece atriyoventriküler blok tanımını karşılıyor mu?

<details><summary><b>Prompt yolu</b>, Alıştırma G: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: Defterde `measure_ecg(leads, fs)` adlı bir fonksiyon tanımlıdır. Fonksiyon, on iki derivasyonu milivolt cinsinden I, II, III, aVR, aVL, aVF, V1-V6 sırasıyla tutan (örnek sayısı, 12) boyutlu bir NumPy dizisini ve Hz cinsinden örnekleme frekansını alır. "hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms" ve "qtc_fridericia_ms" anahtarlarını içeren bir sözlük döndürür. `vclinic.patient_ecg(clinic, patient_id)` bir EKG nesnesi döndürür; bu nesneye `e` diyelim. `e.leads` on iki derivasyonu tutan bir pandas DataFrame'idir, `e.fs` ise Hz cinsinden örnekleme frekansıdır.
Görev: "VC-0188" numaralı hastanın PR aralığını ölç ve birinci derece atriyoventriküler blok tanımını, yani 200 ms'nin üstünde bir PR aralığını, karşılayıp karşılamadığına karar ver.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: PR aralığını en yakın milisaniyeye yuvarla ve kararı yazdır. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma G", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Aynı kodu kayıttaki PR aralığı 190 ms olan "VC-0317" numaralı hasta için de çalıştır; ölçülen değer 190 ms'den en çok 10 ms sapmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma G", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
e = vclinic.patient_ecg(clinic, "VC-0188")
answer = round(measure_ecg(e.leads.to_numpy(), e.fs)["pr_ms"])
teach.check("Alıştırma G", answer)
print("Birinci derece blok (PR > 200 ms):", answer > 200)

**İleri yol.** 2009 önerilerine göre QT, en uzun QT'yi gösteren derivasyonda ölçülmelidir; bu çoğu zaman V2 ya da V3'tür. `measure_ecg` fonksiyonunu, QT'yi II, V2, V3 ve V5 derivasyonlarında ölçüp en uzununu verecek biçimde genişletin. VC-0018 üzerinde deneyin ve her derivasyonun T dalgasına bakın: T dalgası QRS'ye zıt olduğunda teğet yöntemi neden zorlanmaktadır?

In [ ]:
# İleri yol: dört derivasyonda QT
e = vclinic.patient_ecg(clinic, "VC-0018")
for lead in ["II", "V2", "V3", "V5"]:
    print(lead, round(measure_ecg(e.leads.to_numpy(), e.fs, lead=lead)["qt_ms"], 1))
print("gerçek QT:", e.truth["qt_ms"])

**Prompt görevi, 5. bölüm: Kalp hızları boyunca Bazett ve Fridericia.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: ecg_hr (bpm cinsinden kalp hızı) ve ecg_qt (ms cinsinden QT aralığı) sütunlarını içeren `clinic` DataFrame'i.
Görev: İlk 50 hasta için QTc'yi Bazett (QT / √RR) ve Fridericia (QT / RR^(1/3)) formülleriyle hesapla; saniye cinsinden RR aralığını RR = 60 / ecg_hr ile bul. Bazett eksi Fridericia farkını kalp hızına karşı çiz, sıfır düzeyine yatay bir çizgi ekle ve en büyük farkı yazdır.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve bir sayı. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Tam 60 bpm'de iki formül aynı QTc'yi verir; dolayısıyla fark sıfırdır.
</pre>

</details>

**Beklenen sonuç.** Fark, 60 bpm'nin üstündeki her hastada pozitif, 60 bpm'nin altındaki üç hastada negatiftir. En büyük fark 48,6 ms'dir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 6. Gerçek bir kayıtta her atımı bulmak: MIT-BIH 100 numaralı kayıt

*Ders: [Her atımı bulmak: R tepesi tespiti](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#finding-every-beat-r-peak-detection)*

**Formüllerden koda.** Toplama penceresinin genişliği $N = 0.150 \cdot f_s$, `int(round(0.150 * fs))` ile hesaplanır ve 360 Hz'de 54 örnektir. Duyarlılık $Se = TP / (TP + FN)$ ve pozitif öngörü değeri $PPV = TP / (TP + FP)$, `vclinic.measure.match_beats` fonksiyonunun döndürdüğü sayılardan hesaplanır. Bulunan bir atım, bir referans atımına 150 ms'den daha yakınsa doğru sayılır.

Massachusetts Institute of Technology ve Beth Israel Hospital (MIT-BIH) Aritmi Veri Tabanı'nın 100 numaralı kaydı 30 dakika sürer ve 360 Hz'de iki kanal içerir. Kardiyologlar her atımı etiketlemiştir. Kayıt ders dosyalarıyla birlikte gelir; bu yüzden bölüm internet bağlantısı olmadan çalışır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
from vclinic.datasets import read_physionet, read_physionet_annotation
from vclinic.measure import match_beats

mit, fs_mit, notes = read_physionet("100", "mitdb")
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
print(mit.shape, fs_mit, notes)
print(pd.Series(ann_symbols).value_counts())
mit.head()

Etiket sembolleri PhysioNet geleneğini izler: `N` normal atım, `A` atriyal erken atım, `V` ventriküler erken atımdır. `+` bir atım değildir; ritim değişikliğini gösterir. `wfdb` paketi kuruluysa kaydı o okur; kurulu değilse `vclinic.physionet` içindeki küçük okuyucu aynı işi yapar.

In [ ]:
# Dedektörün dört aşaması, ilk 3 saniyede
x = mit["MLII"].to_numpy()
seg = slice(0, int(3 * fs_mit))
y = bandpass(x, fs_mit, 5, 15)
z = (np.gradient(y) * fs_mit) ** 2
w = int(round(0.150 * fs_mit))
m = np.convolve(z, np.ones(w) / w, mode="same")
t = np.arange(len(x)) / fs_mit
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
names = ["ham MLII derivasyonu (mV)", "bant geçiren 5-15 Hz",
         "eğimin karesi", f"hareketli pencere ({w} örnek)"]
for ax, sig, name in zip(axes, [x, y, z, m], names):
    ax.plot(t[seg], sig[seg], color="#9B1B1B" if ax is axes[3] else "k", lw=0.9)
    ax.set_ylabel(name, fontsize=8)
axes[3].set_xlabel("Zaman (s)")
plt.tight_layout()
plt.show()

In [ ]:
# Tüm kayıtta tespit ve etiketlerle karşılaştırma
beat_symbols = set("NLRBAaJSVrFejnE/fQ?")
ref = np.array([s for s, sym in zip(ann_samples, ann_symbols) if sym in beat_symbols])
found = detect_r_peaks(x, fs_mit)
tp, fp, fn = match_beats(found / fs_mit, ref / fs_mit, window_s=0.150)
print(f"referans atımlar: {len(ref)}, bulunan: {len(found)}")
print(f"TP = {tp}, FP = {fp}, FN = {fn}")
print(f"Se = {100 * tp / (tp + fn):.2f} %, PPV = {100 * tp / (tp + fp):.2f} %")

**Alıştırma H.** 100 numaralı kayıtta kaç atriyal erken atım (`A` sembolü) vardır?

<details><summary><b>Prompt yolu</b>, Alıştırma H: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `ann_symbols` adlı bir Python listesi, MIT-BIH 100 numaralı kaydın etiket sembollerini (her etiket için bir sembol) tutar: normal atım için "N", atriyal erken atım için "A", ventriküler erken atım için "V" ve ritim değişikliği için "+".
Görev: Atriyal erken atımları say.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Atım sayısı, tam sayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma H", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Her sembolü collections.Counter ile say ve sayıları yazdır; toplamları len(ann_symbols) değerine eşit olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma H", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = ann_symbols.count("A")
teach.check("Alıştırma H", answer)

**Gözlemle ve düşün.** Dedektör 100 numaralı kaydın bütün atımlarını buluyor. Bu sonuç, dedektöre her kayıtta güvenmek için neden yeterli değildir? Gürültüyü, geniş ventriküler atımları ve yüksek T dalgalarını düşünün. Dedektörü zorlamak için bir test setinde hangi kayıtları seçerdiniz?

**İleri yol.** İnternet bağlantısıyla `wfdb.rdrecord("105", pn_dir="mitdb")` ve `wfdb.rdann("105", "atr", pn_dir="mitdb")`, gürültüsüyle bilinen 105 numaralı kaydı doğrudan PhysioNet'ten okur. Dedektörü bu kayıtta çalıştırın ve neurokit2 kuruluysa sonucu `neurokit2.ecg_peaks` ile karşılaştırın.

In [ ]:
# İleri yol: PhysioNet'ten 105 numaralı kayıt (internet ve wfdb gerekir)
try:
    import wfdb
    rec = wfdb.rdrecord("105", pn_dir="mitdb", sampto=360 * 300)
    ann = wfdb.rdann("105", "atr", pn_dir="mitdb", sampto=360 * 300)
    ref105 = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in beat_symbols])
    found105 = detect_r_peaks(rec.p_signal[:, 0], rec.fs)
    print(match_beats(found105 / rec.fs, ref105 / rec.fs))
except Exception as err:
    print("Burada kullanılamıyor:", type(err).__name__)

**Prompt görevi, 6. bölüm: Otuz dakika boyunca kalp hızı.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `found`, MIT-BIH 100 numaralı kayıtta bulunan R tepelerini örnek konumları olarak, `fs_mit` ise kaydın örnekleme frekansını (360 Hz) tutar.
Görev: Her atımın kalp hızını 60 / RR ile hesapla (RR saniye cinsinden) ve zamana karşı çiz; zaman ekseni dakika cinsinden olsun. Ortalama kalp hızını yazdır.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve bir sayı. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 360 Hz'de 360 örneklik bir RR aralığı 1 s, yani 60 bpm'dir.
</pre>

</details>

**Beklenen sonuç.** Ortalama kalp hızı yaklaşık 75,8 bpm'dir; atımdan atıma hız yaklaşık 53 ile 115 bpm arasında değişir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 7. Kalp hızı değişkenliği

*Ders: [Kalp hızı değişkenliği](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#heart-rate-variability)*

**Formüllerden koda.** RR aralıkları milisaniye cinsinden bir `rr_ms` dizisindeyse normalden normale aralıkların standart sapması $SDNN = \sqrt{\frac{1}{N-1}\sum (RR_i - \overline{RR})^2}$, `np.std(rr_ms, ddof=1)` ile hesaplanır. Ardışık farkların karelerinin ortalamasının karekökü $RMSSD = \sqrt{\frac{1}{N-1}\sum (RR_{i+1} - RR_i)^2}$ ise `np.sqrt(np.mean(np.diff(rr_ms) ** 2))` ile bulunur.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
# Dersteki çözümlü örnek: altı RR aralığı (ms)
rr_ms = np.array([812, 845, 830, 798, 860, 841])
print("ortalama RR:", rr_ms.mean())
print("SDNN:", round(np.std(rr_ms, ddof=1), 2), "ms")
print("ardışık farklar:", np.diff(rr_ms))
print("RMSSD:", round(np.sqrt(np.mean(np.diff(rr_ms) ** 2)), 2), "ms")

In [ ]:
# Vaka için beş dakikalık D2 kaydı; 1996 Görev Grubu kısa süreli KHD için beş dakikayı önermektedir
from vclinic.ecg import lead_ii_long

t5, x5, truth5 = lead_ii_long(case.params, duration_s=300, fs=250, seed=17)
r5 = detect_r_peaks(x5, 250)
rr5 = np.diff(r5) / 250 * 1000                 # ms cinsinden RR aralıkları
sdnn = np.std(rr5, ddof=1)
rmssd = np.sqrt(np.mean(np.diff(rr5) ** 2))
print(f"atım: {len(r5)}, ortalama kalp hızı: {60000 / rr5.mean():.1f} bpm, SDNN: {sdnn:.1f} ms, RMSSD: {rmssd:.1f} ms")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(r5[1:] / 250, rr5, color="#9B1B1B", lw=0.8)
axes[0].set_xlabel("Zaman (s)")
axes[0].set_ylabel("RR (ms)")
axes[0].set_title("Takogram")
axes[1].scatter(rr5[:-1], rr5[1:], s=6, color="#1F5F8B")
axes[1].set_xlabel("RR$_i$ (ms)")
axes[1].set_ylabel("RR$_{i+1}$ (ms)")
axes[1].set_title("Poincaré grafiği")
plt.tight_layout()
plt.show()

In [ ]:
# 100 numaralı kayıt: ilk 5 dakikanın normalden normale aralıkları
pairs = [(s1, s2) for (s1, a), (s2, b) in zip(zip(ann_samples, ann_symbols),
                                               zip(ann_samples[1:], ann_symbols[1:]))
         if a == "N" and b == "N" and s2 <= 300 * fs_mit]
nn_ms = np.array([(s2 - s1) / fs_mit * 1000 for s1, s2 in pairs])
print(f"NN aralık: {len(nn_ms)}, ortalama kalp hızı: {60000 / nn_ms.mean():.1f} bpm")
print(f"SDNN: {np.std(nn_ms, ddof=1):.1f} ms, RMSSD: {np.sqrt(np.mean(np.diff(nn_ms) ** 2)):.1f} ms")

**Keşfet.** Sanal kliniğin sinyal üreticisinde `hrv` adlı bir değişkenlik parametresi vardır: 1,0 değeri istirahatteki tipik bir yetişkini temsil eder; daha küçük değerler ileri yaşı ya da kalp yetersizliğini taklit eder. Parametreyi değiştirin ve beş dakikalık bir kaydın SDNN ve RMSSD değerlerini izleyin.

In [ ]:
# Keşfet: değişkenlik ve iki ölçüt
def show_hrv(hrv=1.0, heart_rate=70):
    params = dict(case.params, hrv=hrv, hr_bpm=heart_rate)
    _, xs, _ = lead_ii_long(params, duration_s=300, fs=250, seed=5)
    rr = np.diff(detect_r_peaks(xs, 250)) / 250 * 1000
    print(f"hrv = {hrv:.2f}, HR = {heart_rate} bpm -> SDNN = {np.std(rr, ddof=1):.1f} ms, "
          f"RMSSD = {np.sqrt(np.mean(np.diff(rr) ** 2)):.1f} ms")


try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(show_hrv, hrv=FloatSlider(value=1.0, min=0.2, max=1.6, step=0.1),
             heart_rate=IntSlider(value=70, min=50, max=110, step=5, description="HR"))
except ImportError:
    show_hrv(1.0, 70)
    show_hrv(0.4, 70)

**Alıştırma I.** 1000, 960, 1010, 990 ve 1030 ms'lik RR aralıklarının RMSSD değerini hesaplayın ve iki ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma I: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: 1000, 960, 1010, 990 ve 1030 ms'lik RR aralıkları.
Görev: Ardışık farkların karelerinin ortalamasının karekökünü (RMSSD) hesapla: Komşu aralıklar arasındaki farkları al, karelerini al, karelerin ortalamasını bul ve karekökünü al.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: RMSSD'yi iki ondalığa yuvarla. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma I", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 800, 800 ve 800 ms'lik aralıklar 0; 800, 820 ve 800 ms'lik aralıklar ise 20 vermelidir.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma I", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
rr_ex = np.array([1000, 960, 1010, 990, 1030])
answer = round(float(np.sqrt(np.mean(np.diff(rr_ex) ** 2))), 2)
teach.check("Alıştırma I", answer)

**Gözlemle ve düşün.** RMSSD daha çok atımdan atıma olan hızlı değişimlere, SDNN ise yavaş değişimlere de duyarlıdır. 1996 Görev Grubu kısa süreli ölçütler için neden 5 dakikalık kayıtları önermektedir ve farklı uzunluktaki kayıtlardan elde edilen değerler neden karşılaştırılmamalıdır?

**Prompt görevi, 7. bölüm: Bir Poincaré grafiği.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `rr5`, haftanın vakasındaki D2 derivasyonunun beş dakikasına ait RR aralıklarını milisaniye cinsinden tutar.
Görev: Her RR aralığını bir sonrakine karşı çizen bir Poincaré grafiği oluştur; eksenler eşit ölçekli olsun ve y = x doğrusu da çizilsin. SDNN'yi (ddof=1 ile standart sapma) ve RMSSD'yi yazdır.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve iki sayı. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 800, 820 ve 800 ms'lik aralıklar için RMSSD 20 ms'dir.
</pre>

</details>

**Beklenen sonuç.** SDNN 17,7 ms, RMSSD 17,4 ms çıkar; bunlar 7. bölümde yazdırılan değerlerle aynıdır.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 8. PTB Diagnostic veri tabanından gerçek bir 12 derivasyonlu EKG

*Ders: [Gerçek veri: PhysioNet kayıtları](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#real-data-physionet-records)*

Almanya'daki Physikalisch-Technische Bundesanstalt (PTB) kurumunun PTB Diagnostic EKG Veri Tabanı'nda 1 numaralı hastaya ait `s0010_re` kaydı, 12 standart derivasyonu ve üç Frank derivasyonunu 1000 Hz'de 38,4 saniye boyunca içerir. Kaydın başlık dosyasında hastanın klinik özeti bulunur. Bu kayıtla 5. bölümdeki adımların gerçek bir kayıtta nasıl çalıştığı görülmektedir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001")
print(ptb.shape, fs_ptb)
for line in header[:9]:
    print(line)

In [ ]:
# Gerçek kaydın on iki derivasyonu (sütun adları küçük harfle yazılmıştır)
names = {"I": "i", "II": "ii", "III": "iii", "aVR": "avr", "aVL": "avl", "aVF": "avf",
         "V1": "v1", "V2": "v2", "V3": "v3", "V4": "v4", "V5": "v5", "V6": "v6"}
fig = teach.plot_12lead(ptb.index.to_numpy(), ptb, title="PTB patient001, s0010_re",
                        v_range=(-1.5, 1.5), names=names)
plt.show()

In [ ]:
# Gerçek veride Einthoven ve Goldberger ilişkileri
print("max |II - (I + III)| =", round(float(np.abs(ptb["ii"] - (ptb["i"] + ptb["iii"])).max()), 4), "mV")
print("max |aVR + aVL + aVF| =", round(float(np.abs(ptb["avr"] + ptb["avl"] + ptb["avf"]).max()), 4), "mV")

In [ ]:
# Bölüm 5'teki ölçüm gerçek kayıtta
leads_ptb = ptb[[names[k] for k in vclinic.LEADS]].to_numpy()
m_ptb = measure_ecg(leads_ptb, fs_ptb)
pd.Series(m_ptb).round(1)

**Gözlemle ve düşün.** Algoritma bu kayıtta yaklaşık 137 ms'lik bir QRS süresi ölçüyor. Şekildeki V1 derivasyonuna ve inferior derivasyonlara bakın. Bu genişlikte bir QRS'yi hangi örüntü açıklayabilir ve inferior derivasyonlar ne gösteriyor? 2009 önerileri bilgisayar ölçümlerinin gözle denetlenmesini istemektedir: Şekil üzerinde bir atımı kendiniz ölçün ve karşılaştırın.

**İleri yol.** `vx`, `vy` ve `vz` Frank derivasyonları kalp vektörünü neredeyse doğrudan kaydeder. Bir atımın QRS halkasını üç boyutta çizin. Ders kitapları Frank derivasyonları için farklı işaret kuralları kullandığından halka burada kaydedildiği gibi çizilmiştir.

In [ ]:
# İleri yol: Frank derivasyonlarından bir atımın QRS halkası
r_ptb = detect_r_peaks(ptb["ii"].to_numpy(), fs_ptb)
k = r_ptb[3]
loop = ptb[["vx", "vy", "vz"]].to_numpy()[k - int(0.06 * fs_ptb):k + int(0.08 * fs_ptb)]
fig = plt.figure(figsize=(5.5, 5))
ax = fig.add_subplot(projection="3d")
ax.plot(loop[:, 0], loop[:, 1], loop[:, 2], color="#9B1B1B")
ax.set_xlabel("vx (mV)")
ax.set_ylabel("vy (mV)")
ax.set_zlabel("vz (mV)")
plt.show()

**Prompt görevi, 8. bölüm: Gerçek bir enfarktüsün göğüs derivasyonları.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: `ptb`, PTB s0010_re kaydını tutan bir pandas DataFrame'idir. v1-v6 sütunları göğüs derivasyonlarını milivolt cinsinden tutar; örnekleme frekansı `fs_ptb`, yani 1000 Hz'dir.
Görev: V1-V6'nın ilk 3 saniyesini, saniye cinsinden ortak bir zaman ekseni ve milivolt cinsinden voltajla alt alta çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Altı satırlı bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 1000 Hz'de üç saniye derivasyon başına 3000 örnek içerir.
</pre>

</details>

**Beklenen sonuç.** Şekil, her biri 3 s'lik altı satırdan oluşur. Kaydın başlığı akut inferolateral bir enfarktüs belirtir: Lateral duvara hangi göğüs derivasyonları bakar?

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 9. YZ ile eşli kodlama: Kod güvenlik kontrol listesi

*Ders: [YZ ile eşli kodlama, bölüm 1: Kod güvenlik kontrol listesi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#ai-pair-programming-part-1-the-code-safety-checklist)*

Bir YZ asistanı bu eğitimdeki kodların çoğunu yazabilir; ama hatasız çalışıp yanlış bir sayı döndüren kod da yazabilir. Bu yüzden eğitimde YZ'nin yazdığı kod, ekibe yeni katılmış bir üyenin işi gibi ele alınır: Hoş karşılanır ve denetlenir. Aşağıdaki prompt kartı iyi bir promptun yapısını göstermektedir.

**Prompt kartı.** *Rol:* Python öğrenen bir kardiyoloğa yardım ediyorsun. *Bağlam:* NumPy'nin np adıyla içe aktarıldığı bir Jupyter defteri. *Veri:* 500 Hz'de örneklenmiş, 10 saniyelik, 12 derivasyonlu bir EKG; `leads` adlı pandas DataFrame'inde her derivasyon bir sütundur ve değerler milivolt cinsindendir. QT ve RR aralıkları örnek sayısı olarak verilir. *Görev:* Örnek sayısı olarak verilen QT ve RR aralıklarından Bazett formülüyle QTc'yi döndüren bir fonksiyon yaz ve fonksiyonu QT = 196 örnek, RR = 353 örnek için çalıştır. *Kısıtlar:* Yalnızca NumPy kullan; örnekleme frekansını argüman olarak al ve ona varsayılan değer verme. *Çıktı:* Milisaniye cinsinden, bir ondalığa yuvarlanmış QTc. *Test:* 500 Hz'de QT = 200 örnek ve RR = 500 örnek 400,0 ms vermeli; çünkü bu durumda QT 400 ms, RR aralığı 1 s'dir.

**Kullanmadan önce denetleyin.** Ders sayfası, eğitimin kod güvenlik kontrol listesini verir: Çalıştırmadan önce *giriş* (birimler, örnekleme frekansı, derivasyon, hasta tanımlayıcısı yok), çalışırken *mola* (hatalar, fizyolojik değerler) ve kullanmadan önce *çıkış* (bilinen bir gerçek değerle karşılaştırma, her satırın açıklanması, promptun ve değişikliklerin kaydı).

Aşağıdaki fonksiyonu bir YZ asistanı, örnekleme frekansını belirtmeyen daha kısa bir promptla yazmıştır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
# Kısa bir prompttan YZ asistanının yazdığı kod
def qtc_from_samples(qt_samples, rr_samples, fs=360):
    qt_ms = qt_samples / fs * 1000
    rr_ms = rr_samples / fs * 1000
    return qt_ms / np.sqrt(rr_ms)


print(qtc_from_samples(196, 353))

**Gözlemle ve düşün.** Fonksiyon hatasız çalışıyor. Kontrol listesini adım adım uygulayın: Sonuçtaki sorunu hangi duraklamada fark edersiniz ve koddaki iki hata nedir? Hatalardan biri, promptun hiç istemediği varsayılan bir değerden kaynaklanmaktadır.

**Alıştırma J.** Fonksiyonu düzeltin: `fs` değerini varsayılan değer olmadan alın ve RR aralığını saniye cinsinden kullanın. 500 Hz'de 196 ve 353 örnek için QTc değerini hesaplayın ve bir ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma J: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: fs Hz'de örneklenmiş bir EKG; QT ve RR aralıkları örnek sayısı olarak verilmiştir.
Görev: Bir `qtc_from_samples_fixed(qt_samples, rr_samples, fs)` fonksiyonu yaz; fs parametresine varsayılan değer verme. Fonksiyon QT aralığını qt_samples / fs × 1000 ile milisaniyeye, RR aralığını rr_samples / fs ile saniyeye çevirsin ve Bazett formülüyle (QTc = QT / √RR) hesaplanan QTc'yi milisaniye cinsinden döndürsün. Fonksiyonu 500 Hz'de 196 ve 353 örnek için kullan.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: QTc'yi bir ondalığa yuvarla. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma J", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 500 Hz'de 200 ve 500 örnek için QT 400 ms, RR aralığı 1 s olduğundan fonksiyon 400,0 döndürmelidir.
</pre>

</details>

In [ ]:
def qtc_from_samples_fixed(qt_samples, rr_samples, fs):
    return None   # düzeltilmiş formülünüz


answer = qtc_from_samples_fixed(196, 353, 500)
answer = round(answer, 1) if answer is not None else None
teach.check("Alıştırma J", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
def qtc_from_samples_fixed(qt_samples, rr_samples, fs):
    """fs (Hz) frekansında örnek sayısı olarak verilen QT ve RR aralıklarından Bazett QTc (ms)."""
    qt_ms = qt_samples / fs * 1000
    rr_s = rr_samples / fs
    return qt_ms / np.sqrt(rr_s)


answer = round(qtc_from_samples_fixed(196, 353, 500), 1)
teach.check("Alıştırma J", answer)

**YZ kullanım günlüğü.** Her haftalık projeye, bir YZ asistanının her kullanımı için bir satır içeren kısa bir günlük eklenir:

| Araç | Prompt (kısaca) | Neyin denetlendiği | Neyin değiştirildiği |
|---|---|---|---|
| (asistanın adı) | Örnek sayılarından Bazett QTc | 466,5 ms sonucunun VC-0317'nin gerçek değerleriyle karşılaştırılması | Varsayılan fs kaldırıldı; RR saniyeye çevrildi |

**Prompt görevi, 9. bölüm: YZ'nin yazdığı koda ikinci görüş.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot da plt adıyla içe aktarılmıştır; vclinic ders paketi ve teach modülü de kullanılabilir.
Veri: Bu bölümde bir YZ asistanının kısa bir promptla yazdığı `qtc_from_samples` fonksiyonu. Fonksiyonun kodunu bu promptun altına yapıştırın.
Görev: Fonksiyonu üç soruyla gözden geçir. Birimler doğru mu? Örnekleme frekansı doğru kullanılıyor mu? 500 Hz'de 196 örneklik bir QT ve 353 örneklik bir RR aralığı için sonuç fizyolojik bir değer mi? Ardından düzeltilmiş bir fonksiyon ver.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Üç kısa maddelik bir değerlendirme ve düzeltilmiş fonksiyon. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Düzeltilmiş fonksiyon 500 Hz'de 200 ve 500 örnek için 400,0 ms döndürmelidir.
</pre>

</details>

**Beklenen sonuç.** Değerlendirme iki sorunu bulmalıdır: RR aralığının saniye yerine milisaniye cinsinden kullanılması ve 360 Hz'lik varsayılan örnekleme frekansı.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 10. Uygulama anahtarı: Başka bir hasta, başka bir derivasyon

`PATIENT` ve `LEAD` değerlerini değiştirin ve hücreyi yeniden çalıştırın. Bakmaya değer hastalar: VC-0183 (uzun QT aralığı), VC-0018 (sol dal bloğu), VC-0207 (kayıtta 218 ms'lik bir PR aralığı) ve `ecg_rhythm` değeri `AF` olan herhangi bir hasta. `clinic[clinic["stemi_territory"] != "none"]`, 5. haftanın konusu olan akut ST yükselmeli miyokart enfarktüsü (STEMI) tanılı hastaları listeler.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
PATIENT = "VC-0183"
LEAD = "II"

e = vclinic.patient_ecg(clinic, PATIENT)
row = clinic[clinic["patient_id"] == PATIENT].iloc[0]
print(PATIENT, row["age"], row["sex"], row["main_diagnosis"], row["ecg_rhythm"], row["ecg_conduction"])
m = measure_ecg(e.leads.to_numpy(), e.fs, lead=LEAD, sinus=e.truth["rhythm"] == "sinus")
keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
display(pd.DataFrame({"ölçülen": [m[k] for k in keys],
                      "gerçek": [e.truth[k] for k in keys]}, index=keys).round(1))
fig = teach.plot_12lead(e.time, e.leads, title=PATIENT)
plt.show()

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Elektrofizyoloji | VC-0183'ün EKG'sini 50 ile 110 bpm arasındaki kalp hızlarında üretin (`e.params` içindeki `hr_bpm` değerini değiştirin ve `qt_ms` değerini $RR^{1/3}$ ile ölçekleyin). Bazett ve Fridericia QTc değerlerini kalp hızına göre çizin ve bir QT izlem protokolünde hangisinin raporlanacağına karar verin. |
| Girişimsel kardiyoloji | STEMI bölgesi olan hastaları listeleyin, 12 derivasyonlu EKG'lerini çizin ve her bölge için ST yükselmesi olan derivasyonları adlandırın. Bölgenin işaret ettiği suçlu arterle karşılaştırın. |
| Kalp yetersizliği | Klinikteki HFrEF hastalarıyla kalp hastalığı olmayan hastalar arasında 5 dakikalık kayıtların SDNN ve RMSSD değerlerini karşılaştırın. |
| Kardiyo-onkoloji | QT'yi uzatan bir tedavi alan bir hasta için, farklı kalp hızlarındaki seri EKG'lerle bir QTc izlem tablosu oluşturun ve QTc'nin yerel protokolünüzdeki eşiği aştığı vizitleri işaretleyin. |
| Pediatrik kardiyoloji | 2009 önerileri çocuklarda daha düşük QRS eşikleri kullanmaktadır. Kendi yazacağınız bir fonksiyonda geniş QRS eşiğini değiştirin ve bir çocuğun EKG'sinde başka nelerin değişeceğini tartışın. |
| Spor kardiyolojisi | Kalp hızı 45 bpm olan bir istirahat EKG'si üretin ve iki formülle QTc değerlerini karşılaştırın: Hangisi bir sporcuyu anormal olarak etiketleyebilir? |
| Yoğun bakım | `noise="clinic"` ile 5 dakikalık bir kayıt üretin ve R tepesi dedektörünün Se ve PPV değerlerini gerçek atım zamanlarına göre ölçün. |

## Defterden sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma metnini yazın ve öğrenme günlüğünü dışa aktarın. [Hafta sayfasında](README.md) anlatılan 1. haftanın haftalık projesi, 5. bölümdeki `measure_ecg` fonksiyonundan başlar.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>